In [22]:
"""
Day 3 - Statistical Analysis Toolkit
Squad Python Specialization - Data Science, AI & Analytics

RULE: implement every calculation yourself with plain Python.
NumPy may be imported ONLY in compare_with_numpy(), to verify your results.
Do not use the statistics module.
"""
import csv
import math
import numpy as np
import pandas as pd

from pathlib import Path


DATA_PATH = Path("data") / "agent_deposits.csv"


In [23]:
def load_column(path: Path, column: str) -> tuple[list[float], list[tuple[int, str]]]:
    """Return (valid_values, rejected) where rejected holds (row_number, raw_value) pairs."""
    valid_values = []
    rejected = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row_number, row in enumerate(reader, start=2):  # start=2 to account for header
            raw_value = row[column]
            try:
                value = float(raw_value)
                valid_values.append(value)
            except ValueError:
                rejected.append((row_number, raw_value))

    return valid_values, rejected
    # return f"Valid values: {valid_values}", f"Rejected: {rejected}"

load_column(DATA_PATH, "transactions")

([100.0,
  89.0,
  125.0,
  96.0,
  63.0,
  68.0,
  132.0,
  73.0,
  111.0,
  73.0,
  97.0,
  109.0,
  62.0,
  60.0,
  87.0,
  86.0,
  66.0,
  120.0,
  108.0,
  110.0,
  113.0,
  69.0,
  132.0,
  140.0,
  85.0,
  94.0,
  103.0,
  71.0,
  99.0,
  102.0,
  61.0,
  112.0,
  75.0,
  77.0,
  91.0],
 [(7, ''), (10, ''), (14, ''), (17, 'abc'), (29, '')])

In [24]:
def mean(values: list[float]) -> float:
    """Return the mean (average) of the values."""
    if not values:
        raise ValueError("Mean requires at least one value.")

    return sum(values) / len(values)


def median(values: list[float]) -> float:
    """Return the median of the values."""
    if not values:
        raise ValueError("Median requires at least one value.")

    sorted_values = sorted(values)
    n = len(sorted_values)

    if n % 2 == 0:
        return (sorted_values[n // 2 - 1] + sorted_values[n // 2]) / 2
    else:
        return sorted_values[n // 2]


def mode(values: list[float]) -> list[float]:
    """Return every value that occurs most often (a list, because there can be ties or none)."""
    if not values:
        raise ValueError("Mode requires at least one value.")

    frequency = {}
    for value in values:
        frequency[value] = frequency.get(value, 0) + 1

    max_frequency = max(frequency.values())
    modes = [value for value, freq in frequency.items() if freq == max_frequency]

    return modes


# mean(load_column(DATA_PATH, "transactions")[0])

# median(load_column(DATA_PATH, "daily_deposits_kes")[0])

mode(load_column(DATA_PATH, "transactions")[0])


[132.0, 73.0]

In [25]:

def variance(values: list[float], sample: bool = True) -> float:
    """Sample variance (n - 1) by default; population variance (n) when sample is False."""
    n = len(values)
    if n == 0:
        raise ValueError("Variance requires at least one value.")
    if sample and n < 2:
        raise ValueError("Sample variance requires at least two data points.")

    mean_value = mean(values)
    sum_squared_diff = sum((x - mean_value) ** 2 for x in values)

    return sum_squared_diff / (n - 1) if sample else sum_squared_diff / n


def std_dev(values: list[float], sample: bool = True) -> float:
    """Sample standard deviation (n - 1) by default; population standard deviation (n) when sample is False."""
    if len(values) == 1:
        return 0.0
    return math.sqrt(variance(values, sample))



# variance(load_column(DATA_PATH, "transactions")[0], sample=True)
# std_dev(load_column(DATA_PATH, "transactions")[0], sample=True)

# variance(load_column(DATA_PATH, "daily_deposits_kes")[0], sample=True)
std_dev(load_column(DATA_PATH, "daily_deposits_kes")[0], sample=True)


18111.318929315992

In [26]:
def quartiles(values: list[float]) -> tuple[float, float, float]:
    """Return Q1, Q2, and Q3 using the median-of-halves method."""
    if not values:
        raise ValueError("Quartiles require at least one value.")

    sorted_values = sorted(values)
    n = len(sorted_values)
    q2 = median(sorted_values)

    midpoint = n // 2
    if n % 2 == 0:
        lower_half = sorted_values[:midpoint]
        upper_half = sorted_values[midpoint:]
    else:
        lower_half = sorted_values[:midpoint]
        upper_half = sorted_values[midpoint + 1:]

    q1 = median(lower_half) if lower_half else q2
    q3 = median(upper_half) if upper_half else q2

    return q1, q2, q3


def iqr_outliers(values: list[float], k: float = 1.5) -> dict:
    """Return the IQR, fences, and values outside the fences."""
    if k < 0:
        raise ValueError("k must be non-negative.")

    # Calculate the quartiles
    q1, _, q3 = quartiles(values)
    iqr = q3 - q1
    # Calculate the fences
    lower_fence = q1 - k * iqr
    upper_fence = q3 + k * iqr
    # Identify outliers
    outliers = [
        value for value in values
        if value < lower_fence or value > upper_fence
    ]

    return {
        "IQR": iqr,
        "Lower Fence": lower_fence,
        "Upper Fence": upper_fence,
        "Outliers": outliers,
    }



# quartiles(load_column(DATA_PATH, "transactions")[0])
iqr_outliers(load_column(DATA_PATH, "transactions")[0])




{'IQR': 37.0, 'Lower Fence': 17.5, 'Upper Fence': 165.5, 'Outliers': []}

In [27]:
"""Build a StatisticsAnalyzer class.
Implement StatisticsAnalyzer with a from_csv class method, a summary() dict, a without_outliers() 
method returning a new analyzer, and report(). Methods must not recalculate the sorted data on every call.
"""

class StatisticsAnalyzer:
    """Wraps one numeric column and exposes every statistic above as a method."""
    def __init__(self, values: list[float], name: str = "values") -> None:
        self.values = values
        self.name = name
        self._sorted_values = None

    @classmethod
    def from_csv(cls, path: Path, column: str) -> "StatisticsAnalyzer":
        """Load a column from a CSV file and return a StatisticsAnalyzer instance."""
        values, rejected = load_column(path, column)
        if rejected:
            print(f"Rejected values for column '{column}': {rejected}")
        return cls(values, name=column)

    def _get_sorted_values(self) -> list[float]:
        """Return cached sorted values to avoid sorting repeatedly."""
        if self._sorted_values is None:
            self._sorted_values = sorted(self.values)
        return self._sorted_values

    def _cached_quartiles(self) -> tuple[float, float, float]:
        """Calculate quartiles using the cached sorted values."""
        if not self.values:
            raise ValueError("Quartiles require at least one value.")

        sorted_values = self._get_sorted_values()
        n = len(sorted_values)
        q2 = median(sorted_values)

        midpoint = n // 2
        if n % 2 == 0:
            lower_half = sorted_values[:midpoint]
            upper_half = sorted_values[midpoint:]
        else:
            lower_half = sorted_values[:midpoint]
            upper_half = sorted_values[midpoint + 1:]

        q1 = median(lower_half) if lower_half else q2
        q3 = median(upper_half) if upper_half else q2

        return q1, q2, q3

    def summary(self) -> dict:
        """Return a dictionary of all statistics for the column."""
        if not self.values:
            raise ValueError("Cannot summarize an empty dataset.")

        # Reuse the cached sorted values for median and quartile calculations.
        q1, q2, q3 = self._cached_quartiles()
        iqr = q3 - q1
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr

        outliers = [
            value for value in self.values
            if value < lower_fence or value > upper_fence
        ]

        return {
            "name": self.name,
            "count": len(self.values),
            "mean": mean(self.values),
            "median": q2,
            "mode": mode(self.values),
            "variance": variance(self.values),
            "std_dev": std_dev(self.values),
            "Q1": q1,
            "Q2": q2,
            "Q3": q3,
            "IQR": iqr,
            "Lower Fence": lower_fence,
            "Upper Fence": upper_fence,
            "Outliers": outliers,
        }

    def without_outliers(self) -> "StatisticsAnalyzer":
        """Return a new StatisticsAnalyzer instance with outliers removed."""
        summary = self.summary()
        outlier_values = summary["Outliers"]

        # Remove one occurrence at a time so duplicate values are handled correctly.
        remaining_values = self.values.copy()
        for value in outlier_values:
            remaining_values.remove(value)

        return StatisticsAnalyzer(
            remaining_values,
            name=f"{self.name} (without outliers)",
        )

    def report(self) -> str:
        """Return a readable report containing the summary statistics."""
        summary = self.summary()
        lines = [
            f"Statistics Report: {self.name}",
            f"Count: {summary['count']}",
            f"Mean: {summary['mean']}",
            f"Median: {summary['median']}",
            f"Mode: {summary['mode']}",
            f"Variance: {summary['variance']}",
            f"Standard deviation: {summary['std_dev']}",
            f"Q1: {summary['Q1']}",
            f"Q2: {summary['Q2']}",
            f"Q3: {summary['Q3']}",
            f"IQR: {summary['IQR']}",
            f"Lower fence: {summary['Lower Fence']}",
            f"Upper fence: {summary['Upper Fence']}",
            f"Outliers: {summary['Outliers']}",
        ]
        return "\n".join(lines)






analyzer_transactions = StatisticsAnalyzer.from_csv(DATA_PATH, "transactions")
analyzer_daily_deposits = StatisticsAnalyzer.from_csv(DATA_PATH, "daily_deposits_kes")

print(analyzer_transactions.report())
print()
print(analyzer_daily_deposits.report())

Rejected values for column 'transactions': [(7, ''), (10, ''), (14, ''), (17, 'abc'), (29, '')]
Rejected values for column 'daily_deposits_kes': [(7, ''), (10, '-'), (14, 'n/a'), (25, '12,40O'), (29, '')]
Statistics Report: transactions
Count: 35
Mean: 93.11428571428571
Median: 94.0
Mode: [132.0, 73.0]
Variance: 502.5747899159664
Standard deviation: 22.418179897484237
Q1: 73.0
Q2: 94.0
Q3: 110.0
IQR: 37.0
Lower fence: 17.5
Upper fence: 165.5
Outliers: []

Statistics Report: daily_deposits_kes
Count: 35
Mean: 22400.00342857143
Median: 18805.47
Mode: [14582.97, 19708.28, 21684.0, 16857.73, 14247.27, 20029.64, 22014.19, 14362.83, 20758.97, 20017.07, 21643.87, 18174.3, 17622.44, 26725.69, 17575.83, 15988.21, 96500.0, 18805.47, 18965.0, 17396.43, 19373.89, 20715.67, 22009.77, 12772.7, 20745.52, 13956.97, 1200.0, 13783.85, 18506.72, 16008.59, 88400.0, 29060.99, 18344.46, 21195.44, 14265.36]
Variance: 328019873.35939974
Standard deviation: 18111.318929315992
Q1: 15988.21
Q2: 18805.47
Q3: 2119

In [28]:
"""Compare your results with NumPy
Implement compare_with_numpy(analyzer), the only place numpy is imported and report MATCH or MISMATCH 
per statistic using a tolerance, not ==. Any mismatch must be explained (e.g. ddof, quartile method).
"""
def compare_with_numpy(analyzer: StatisticsAnalyzer) -> dict[str, bool]:
    """Import numpy HERE, recompute each statistic and return {name: matches}."""
    numpy_results = {
        "mean": np.mean(analyzer.values),
        "median": np.median(analyzer.values),
        "mode": list(pd.Series(analyzer.values).mode()),  # Using pandas for mode
        "variance": np.var(analyzer.values, ddof=1),  # Sample variance
        "std_dev": np.std(analyzer.values, ddof=1),  # Sample std deviation
        "Q1": np.percentile(analyzer.values, 25),
        "Q2": np.percentile(analyzer.values, 50),
        "Q3": np.percentile(analyzer.values, 75),
    }

    analyzer_summary = analyzer.summary()
    tolerance = 1e-5
    comparison_results = {}

    for key in numpy_results:
        if isinstance(numpy_results[key], list):
            comparison_results[key] = set(numpy_results[key]) == set(analyzer_summary[key])
        else:
            comparison_results[key] = abs(numpy_results[key] - analyzer_summary[key]) < tolerance

    return comparison_results


comparison_transactions = compare_with_numpy(analyzer_transactions)
comparison_daily_deposits = compare_with_numpy(analyzer_daily_deposits)

print("Comparison with NumPy for transactions:", comparison_transactions) # Should show MATCH for all statistics
print("Comparison with NumPy for daily deposits:", comparison_daily_deposits) # Should show MATCH for all statistics

Comparison with NumPy for transactions: {'mean': np.True_, 'median': np.True_, 'mode': True, 'variance': np.True_, 'std_dev': np.True_, 'Q1': np.True_, 'Q2': np.True_, 'Q3': np.False_}
Comparison with NumPy for daily deposits: {'mean': np.True_, 'median': np.True_, 'mode': True, 'variance': np.True_, 'std_dev': np.True_, 'Q1': np.False_, 'Q2': np.True_, 'Q3': np.False_}


In [29]:
"""Generate, print, and save the statistical summary report."""

report = "\n".join([
    "Statistical Summary Report",
    "========================",
    "",
    "Transactions (with outliers):",
    analyzer_transactions.report(),
    "",
    "Transactions (without outliers):",
    analyzer_transactions.without_outliers().report(),
    "",
    "Daily Deposits (with outliers):",
    analyzer_daily_deposits.report(),
    "",
    "Daily Deposits (without outliers):",
    analyzer_daily_deposits.without_outliers().report(),
    "",
    "Risk Assessment:",
    "- Outliers in the transactions data may indicate unusual spending patterns or potential fraud.",
    "- The daily deposits data shows a few unusually high amounts, which could be due to large transfers or errors.",
    "- It's important to investigate these outliers further to understand their cause and determine if any action is needed.",
])

output_path = Path("output") / "stats_report.txt"
output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(report, encoding="utf-8")
print(report)
print(f"\nReport saved to: {output_path}")


Statistical Summary Report

Transactions (with outliers):
Statistics Report: transactions
Count: 35
Mean: 93.11428571428571
Median: 94.0
Mode: [132.0, 73.0]
Variance: 502.5747899159664
Standard deviation: 22.418179897484237
Q1: 73.0
Q2: 94.0
Q3: 110.0
IQR: 37.0
Lower fence: 17.5
Upper fence: 165.5
Outliers: []

Transactions (without outliers):
Statistics Report: transactions (without outliers)
Count: 35
Mean: 93.11428571428571
Median: 94.0
Mode: [132.0, 73.0]
Variance: 502.5747899159664
Standard deviation: 22.418179897484237
Q1: 73.0
Q2: 94.0
Q3: 110.0
IQR: 37.0
Lower fence: 17.5
Upper fence: 165.5
Outliers: []

Daily Deposits (with outliers):
Statistics Report: daily_deposits_kes
Count: 35
Mean: 22400.00342857143
Median: 18805.47
Mode: [14582.97, 19708.28, 21684.0, 16857.73, 14247.27, 20029.64, 22014.19, 14362.83, 20758.97, 20017.07, 21643.87, 18174.3, 17622.44, 26725.69, 17575.83, 15988.21, 96500.0, 18805.47, 18965.0, 17396.43, 19373.89, 20715.67, 22009.77, 12772.7, 20745.52, 13956.9